# Countdown GRPO — Colab runbook (Qwen3.5-0.8B, free T4)

End-to-end: setup → Drive → config → preflight → train → verify.
Plain Python cells (no `!python -c` quoting traps). Keep this tab foreground while training.

In [ ]:
import os

if os.path.isdir("rl-mini"):
    %cd rl-mini
    !git pull
else:
    !git clone https://github.com/GauthierRoy/rl-mini.git
    %cd rl-mini

!make install-colab
# Qwen3.5 arch (qwen3_5) needs transformers main; requirements pin is older.
!pip install -q -U "git+https://github.com/huggingface/transformers.git@main"


In [ ]:
!pip install -q pandas pyarrow

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

os.makedirs("configs", exist_ok=True)
open("configs/countdown_overnight.yaml", "w").write("""
# Qwen3.5-0.8B countdown (T4 free). Text-only fork of Qwen/Qwen3.5-0.8B:
# same tokenizer/template, no vision encoder. Sampling per model card.
# For 2B instead: model: principled-intelligence/Qwen3.5-2B-text-only,
# per_device_train_batch_size: 1, gradient_accumulation_steps: 8,
# gradient_checkpointing: true, num_tasks: 4000 (see configs/countdown_q35_2b.yaml).
model: principled-intelligence/Qwen3.5-0.8B-text-only
output_dir: /content/drive/MyDrive/rl-mini/countdown-q35-08b
num_tasks: 2000
num_numbers: 4
num_generations: 8
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
max_completion_length: 256
max_steps: 300
learning_rate: 0.000001
beta: 0.001
temperature: 1.0
top_p: 1.0
use_peft: true
lora_r: 32
lora_alpha: 64
lora_target_modules: [q_proj, v_proj]
logging_steps: 5
save_steps: 25
resume: false
gradient_checkpointing: false
""")
print("wrote", os.path.abspath("configs/countdown_overnight.yaml"))


In [ ]:
import sys

sys.path.insert(0, "src")
from envs.countdown import generate_task, make_prompt
from rewards_countdown import correctness_reward, format_reward

t = generate_task(4)
print(t)
print(make_prompt(t["numbers"], t["target"]))
demo = "<think>10-2=8, 8*3=24.</think> <answer>(10-2)*3</answer>"
print("fmt_good:", format_reward([demo]), "fmt_bad:", format_reward(["no tags"]))
print("corr_runs:", correctness_reward([demo], [[10, 2, 3, 7]], [24]))

In [ ]:
!python src/train_countdown.py --config configs/countdown_overnight.yaml

## Gate check (run any time after ~step 10)

Kill gates: `fmt_rate` still 0.0 at step 25+ → prompt bug, stop. Mixed 0.1–0.5 → healthy.

In [ ]:
import glob

import pandas as pd

fs = sorted(glob.glob("/content/drive/MyDrive/rl-mini/countdown-q35-08b/completions/*.parquet"))
f = fs[-1]
df = pd.read_parquet(f)
print(f, len(df))
print("corr_rate:", (df["correctness_reward"] > 0).mean(), "fmt_rate:", (df["format_reward"] > 0).mean())
print("think_rate:", df["completion"].str.contains("<think>", regex=False).mean())

## Morning summary — paste this table back for the training decision

In [ ]:
import glob

import pandas as pd

fs = sorted(glob.glob("/content/drive/MyDrive/rl-mini/countdown-q35-08b/completions/*.parquet"))
for f in fs:
    df = pd.read_parquet(f)
    cr = (df["correctness_reward"] > 0).mean()
    fr = (df["format_reward"] > 0).mean()
    print(f[-22:-8], f"n={len(df)} corr={cr:.2f} fmt={fr:.2f} adv_std={df['advantage'].std():.3f} len={df['completion'].str.len().mean():.0f}")